In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier 

from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score,
accuracy_score,classification_report,confusion_matrix)
import joblib


In [2]:
df = pd.read_csv(r"E:\telecom ml work\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df.head(5).to_string(index=False))

customerID gender  SeniorCitizen Partner Dependents  tenure PhoneService    MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies       Contract PaperlessBilling             PaymentMethod  MonthlyCharges TotalCharges Churn
7590-VHVEG Female              0     Yes         No       1           No No phone service             DSL             No          Yes               No          No          No              No Month-to-month              Yes          Electronic check           29.85        29.85    No
5575-GNVDE   Male              0      No         No      34          Yes               No             DSL            Yes           No              Yes          No          No              No       One year               No              Mailed check           56.95       1889.5    No
3668-QPYBK   Male              0      No         No       2          Yes               No             DSL            Yes          Yes               

In [41]:
df.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [4]:

# Remove unnecessary column
df.drop(columns=["customerID"], inplace=True)


In [5]:

# Convert numerical column
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [6]:

# Separate targets
y_reg = df["MonthlyCharges"]
y_clf = df["Churn"]

# Features: remove both targets
X = df.drop(
    columns=["MonthlyCharges", "Churn"]
)


In [7]:
# Numerical features
num_val = [
    "SeniorCitizen",
    "tenure",
    "TotalCharges"
]

In [8]:
# Categorical features
cat_val = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [9]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer())
])

In [ ]:
cat_pipeline = Pipeline([
     ("imputer", SimpleImputer(strategy="most_frequent")),
     ("ohe", OneHotEncoder(handle_unknown="ignore"))
 ])

In [11]:
 processor = ColumnTransformer([
     ("num", num_pipeline, num_val),
     ("cat", cat_pipeline, cat_val)
 ])

In [ ]:
# regression data h
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)



In [ ]:
# classification data
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

In [14]:
# Regression pipeline
reg_model = Pipeline([
    ("processor", processor),
    ("model", RandomForestRegressor(random_state=42))
])


In [15]:
# Cross-validation
reg_scores = cross_val_score(
    reg_model,
    X_train_reg,
    y_train_reg,
    cv=7,
    scoring="r2"
)

print("Regression CV Scores:", reg_scores)
print("Mean Regression CV Score:", reg_scores.mean())

Regression CV Scores: [0.99781751 0.99772399 0.99755718 0.99784616 0.99814406 0.99797445
 0.99807807]
Mean Regression CV Score: 0.9978773457095004


In [16]:

# Train regression model
reg_model.fit(X_train_reg, y_train_reg)

# Regression predictions
reg_pred = reg_model.predict(X_test_reg)

print("\nREGRESSION RESULTS")
print("MAE:", mean_absolute_error(y_test_reg, reg_pred))
print("MSE:", mean_squared_error(y_test_reg, reg_pred))
print("RMSE:", mean_squared_error(y_test_reg, reg_pred) ** 0.5)
print("R2:", r2_score(y_test_reg, reg_pred))


REGRESSION RESULTS
MAE: 0.9196531819257145
MSE: 1.6641622355886736
RMSE: 1.290024122095658
R2: 0.9981619075759558


In [37]:
joblib.dump(reg_model, "reg_model.pkl")

['reg_model.pkl']

In [17]:
# Classification pipeline
clf_model = Pipeline([
    ("processor", processor),
    ("model", RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])




In [18]:
# Cross-validation
clf_model_scores = cross_val_score(
    reg_model,
    X_train_reg,
    y_train_reg,
    cv=7,
    scoring="r2"
)

print("Regression CV Scores:", reg_scores)
print("Mean Regression CV Score:", reg_scores.mean())

Regression CV Scores: [0.99781751 0.99772399 0.99755718 0.99784616 0.99814406 0.99797445
 0.99807807]
Mean Regression CV Score: 0.9978773457095004


In [19]:
clf_model.fit(X_train_clf, y_train_clf)
clf_pred = clf_model.predict(X_test_clf)

print("\nCLASSIFICATION RESULTS")
print("Accuracy:", accuracy_score(y_test_clf, clf_pred))
print("\nClassification Report:")
print(classification_report(y_test_clf, clf_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_clf, clf_pred))


CLASSIFICATION RESULTS
Accuracy: 0.7785663591199432

Classification Report:
              precision    recall  f1-score   support

          No       0.82      0.89      0.86      1035
         Yes       0.61      0.46      0.52       374

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.69      1409
weighted avg       0.76      0.78      0.77      1409


Confusion Matrix:
[[926 109]
 [203 171]]


In [39]:
joblib.dump(clf_model, "clf_model_churn.pkl")

['clf_model_churn.pkl']